In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from preprocess import TextPreprocessor, create_data_input

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\TTD\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df = pd.read_csv('arxiv_dataset/train.csv')
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission,recency_score
0,0,0,0,cs-9308101v1,Dynamic Backtracking,cs.AI,M. L. Ginsberg,M. L. Ginsberg,Because of their occasional need to return to ...,12006,0.05
1,1,1,1,cs-9308102v1,A Market-Oriented Programming Environment and ...,cs.AI,M. P. Wellman,M. P. Wellman,Market price systems constitute a well-underst...,12006,0.05
2,2,2,2,cs-9309101v1,An Empirical Analysis of Search in GSAT,cs.AI,I. P. Gent; T. Walsh,I. P. Gent,We describe an extensive study of search in GS...,11975,0.05
3,3,3,3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,cs.AI,F. Bergadano; D. Gunetti; U. Trinchero,F. Bergadano,As real logic programmers normally use cut (!)...,11914,0.05
4,4,4,4,cs-9311102v1,Software Agents: Completing Patterns and Const...,cs.AI,J. C. Schlimmer; L. A. Hermens,J. C. Schlimmer,To support the goal of allowing users to recor...,11914,0.05


In [3]:
sbert_embeddings = np.load('arxiv_dataset/SBERTEmbeddings_final.npy')

In [4]:
import scipy.sparse
import joblib

tfidf_matrix = scipy.sparse.load_npz('arxiv_dataset/tfidf_matrix.npz')
preprocessor = joblib.load('arxiv_dataset/text_preprocessor.pkl')

In [5]:
tfidf_matrix.shape

(136238, 10000)

In [6]:
# from sklearn.metrics.pairwise import cosine_similarity as cossim
# pair = (0, 0)
# max = -1.0
# j=3
# for i in range(0, 136237):
#     if max < cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0] and i!=j:
#         max = cossim(tfidf_matrix[j], tfidf_matrix[i])[0][0]
#         pair = (j, i)
# print(max, pair)
    

In [7]:
def build_global_index(df):
    paper_ids = df['paper_id'].values
    
    mappings = {i: pid for i, pid in enumerate(paper_ids)}
    reverse_mapping = {pid: i for i, pid in enumerate(paper_ids)}
    
    return mappings, reverse_mapping
mappings, reverse_mapping = build_global_index(df)
reccency_scores = df['recency_score'].values
# Tốc độ tạo chỉ mất khoảng 0.05 giây
paper_to_cat = pd.Series(df['primary_category'].values, index=df['paper_id']).to_dict()


In [8]:
import json
from sklearn.metrics.pairwise import cosine_similarity as cossin
from sklearn.decomposition import PCA
pca = PCA(n_components=384)
def load_user(path_users='UserDataGenerator/synthetic_users.json', vector_path='user_profiles/user_vectors_halflife_90.npz', sbert=False):
    with open(path_users, 'r') as f:
        data = json.load(f)
    vector_data = np.load(vector_path)
    users_vector = vector_data['vectors']
    if sbert:
        users_vector = pca.fit_transform(users_vector)
    users = data['train']
    users_id = [user['user_id'] for user in users]
    target = {user['user_id'] : user['target_paper'] for user in users}
    negative = {user['user_id'] : user['negative_papers'] for user in users}
    history = {user['user_id'] : [x[0] for x in user['train_history']] for user in users}
    return users_id, target, negative, users_vector, history
        

In [9]:
def history_set(user, history):
    his = set()
    for paper in history[user]:
        his.add(paper_to_cat[paper])
    return his

In [10]:
#create dataloader
#cosine_sim (float) recency_score (float) category_match (int)
def create_data_LTR(embeding_matrix = tfidf_matrix, sbert=False):
    if sbert:
        vector_path = 'user_profiles_SBERT/user_vectors_halflife_90.npz'
    else:
        vector_path = 'user_profiles/user_vectors_halflife_90.npz'
    users_id, target, negative, users_vector, history = load_user(vector_path=vector_path, sbert=sbert)
    print(users_vector.shape)
    print(embeding_matrix.shape)
    data = [] 
    labels = []
    for u, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        target_paper = target[user]
        cos = cossin(users_vector[u].reshape(1,-1), embeding_matrix[reverse_mapping[target_paper]].reshape(1,-1))[0][0]
        rec = reccency_scores[reverse_mapping[target_paper]]
        cat_match = int(paper_to_cat[target_paper] in user_his_cat)
        data.append(np.array([cos, rec, cat_match]))
        labels.append(1)

        for neg in negative[user]:
            cos = cossin(users_vector[u].reshape(1,-1), embeding_matrix[reverse_mapping[neg]].reshape(1,-1))[0][0]
            rec = reccency_scores[reverse_mapping[neg]]
            cat_match = int(paper_to_cat[neg] in user_his_cat)
            data.append(np.array([cos, rec, cat_match]))
            labels.append(0)
    labels = np.array(labels).astype(np.int32) 
    data_train = np.array(data)
    return labels, data_train


In [11]:
labels, data_tfidf = create_data_LTR()
labels, data_sberts = create_data_LTR(embeding_matrix=sbert_embeddings, sbert=True)

(1000, 10000)
(136238, 10000)
(1000, 384)
(136238, 384)


In [12]:
print(data_sberts[0])

[-0.00605355  0.05184123  0.        ]


In [13]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

def train(X_train, y_train):
    # -------------------------------------------------------------------------
    # BƯỚC 1: KHỞI TẠO MÔ HÌNH GRADIENT BOOSTING
    # -------------------------------------------------------------------------
    # Cấu hình các siêu tham số (Hyperparameters) chuẩn cho bài toán LTR nhẹ
    ltr_model = GradientBoostingClassifier(
        n_estimators=500,      # Số lượng cây quyết định dựng nối tiếp nhau
        learning_rate=0.1,     # Tốc độ học (shrinkage) để tránh overfit
        max_depth=3,           # Độ sâu tối đa của mỗi cây (phù hợp với 3-5 features)
        random_state=42,       # Cố định seed để kết quả ổn định sau mỗi lần chạy
        verbose=1              # Bật log để xem tiến trình học của các cây
    )

    # -------------------------------------------------------------------------
    # BƯỚC 2: HUẤN LUYỆN (FIT MODEL)
    # -------------------------------------------------------------------------
    print("Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...")
    # X_train là ma trận dataloader cỡ [N_samples, 3] của ông
    # y_train là mảng nhãn [N_samples] chứa 0 và 1
    ltr_model.fit(X_train, y_train)
    print(" Huấn luyện hoàn tất!")

    # -------------------------------------------------------------------------
    # BƯỚC 3: ĐÁNH GIÁ NHANH ĐỘ CHÍNH XÁC PHÂN LỚP
    # -------------------------------------------------------------------------
    y_pred = ltr_model.predict(X_train)
    print("\n--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TRAIN ---")
    print(f"Accuracy Score: {accuracy_score(y_train, y_pred):.4f}")
    print(classification_report(y_train, y_pred))

    # -------------------------------------------------------------------------
    # BƯỚC 4: XEM TẦM ẢNH HƯỞNG CỦA CÁC ĐẶC TRƯNG (FEATURE IMPORTANCE)
    # -------------------------------------------------------------------------
    # Cái này cực kỳ đắt giá để đưa vào báo cáo/slide đồ án
    importances = ltr_model.feature_importances_
    features_names = ['cosine_sim', 'recency_score', 'category_match']

    print("\n--- TRỌNG SỐ TỰ HỌC ĐƯỢC ĐỐI VỚI TỪNG ĐẶC TRƯNG ---")
    for name, imp in zip(features_names, importances):
        print(f"🔹 {name}: {imp*100:.2f}%")
    return ltr_model

In [14]:
test = pd.read_csv('arxiv_dataset/test.csv')
test.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,paper_id,title,primary_category,authors,first_author,abstract,days_since_submission,recency_score
0,0,0,0,2604.26951v1,Turning the TIDE: Cross-Architecture Distillat...,cs.CL,Gongbo Zhang; Wen Wang; Ye Tian; Li Yuan,Gongbo Zhang,Diffusion large language models (dLLMs) offer ...,47,0.711484
1,1,1,1,2604.26231v1,ProMax: Exploring the Potential of LLM-derived...,cs.IR,Yi Zhang; Yiwen Zhang; Kai Zheng; Tong Chen; H...,Yi Zhang,The remarkable text understanding and generati...,47,0.711484
2,2,2,2,2604.26567v1,AirZoo: A Unified Large-Scale Dataset for Grou...,cs.CV,Xiaoya Cheng; Rouwan Wu; Xinyi Liu; Zeyu Cui; ...,Xiaoya Cheng,Despite the rapid progress in data-driven 3D v...,47,0.711484
3,3,3,3,2604.26565v1,"DenseStep2M: A Scalable, Training-Free Pipelin...",cs.CV,Mingji Ge; Qirui Chen; Zeqian Li; Weidi Xie,Mingji Ge,Long-term video understanding requires interpr...,47,0.711484
4,4,4,4,2604.26520v1,3D-LENS: A 3D Lifting-based Elevated Novel-vie...,cs.CV,William Grolleau; Astrid Sabourin; Guillaume L...,William Grolleau,Aerial-Ground Re-Identification (AG-ReID) is c...,47,0.711484


In [15]:
testing_text = create_data_input(test, columns = ['title', 'abstract', 'primary_category'])
print(len(testing_text))
tfidf_test = preprocessor.transform(testing_text)

7701


In [16]:
val_mapping, reverse_val_mapping = build_global_index(test)
val_rec_scores = test['recency_score'].values
val_paper_to_cat = pd.Series(test['primary_category'].values, index=test['paper_id']).to_dict()

In [17]:
# def create_val_LTR(embeding_matrix = tfidf_test):
#     users_id, target, negative, users_vector, history = load_user()
#     data = [] 
#     for u, user in enumerate(users_id):
#         user_his_cat = history_set(user, history)
#         for pred_paper_idx, pred_paper_embedding in enumerate(embeding_matrix): 
#             cos = cossin(users_vector[u].reshape(1,-1), pred_paper_embedding)[0][0]
#             rec = val_rec_scores[pred_paper_idx]
#             cat_match = int(val_paper_to_cat[pred_paper_idx] in user_his_cat)
#             data.append(np.array([cos, rec, cat_match]))
#     data_train = np.array(data)
#     return data_train

In [18]:
def create_val_LTR(embeding_matrix=tfidf_test):
    users_id, target, negative, users_vector, history = load_user()
    
    num_papers = embeding_matrix.shape[0] 
    all_users_features = []
    
    for u, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        user_tfidf_2d = users_vector[u].reshape(1, -1)
        
        cos_scores = cossin(user_tfidf_2d, embeding_matrix).flatten()
        rec_scores = val_rec_scores 
        
        cat_match_scores = np.zeros(num_papers, dtype=np.int16)
        for idx in range(num_papers):
            if val_paper_to_cat[val_mapping[idx]] in user_his_cat:
                cat_match_scores[idx] = 1
                
        user_features = np.column_stack((cos_scores, rec_scores, cat_match_scores))
        all_users_features.append(user_features)
        
    data_train = np.vstack(all_users_features)
    
    print(" Kích thước ma trận Val LTR cuối cùng:", data_train.shape)
    return data_train
X_pool = create_val_LTR()

 Kích thước ma trận Val LTR cuối cùng: (7701000, 3)


In [19]:
ltr_model_sbert = train(data_sberts, labels)

Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...
      Iter       Train Loss   Remaining Time 
         1           0.8178            1.21m
         2           0.8160           40.25s
         3           0.8144           28.71s
         4           0.8132           22.66s
         5           0.8117           18.68s
         6           0.8107           15.91s
         7           0.8092           14.00s
         8           0.8085           12.50s
         9           0.8073           11.32s
        10           0.8067           10.47s
        20           0.7995            6.24s
        30           0.7936            4.93s
        40           0.7880            4.50s
        50           0.7821            4.18s
        60           0.7763            3.83s
        70           0.7706            3.70s
        80           0.7658            3.52s
        90           0.7599            3.33s
       100           0.7556            3.19s
       200           0.7202            

In [20]:
# Dự đoán xác suất cho toàn bộ danh sách ứng viên
# ltr_model.predict_proba trả về mảng cỡ [N_candidates, 2] (cột 0 là nhãn 0, cột 1 là nhãn 1)
# Ông chỉ lấy cột [:, 1] tức là xác suất bài báo ăn điểm Positive
ltr_model = train(data_tfidf, labels)
predicted_scores = ltr_model.predict_proba(X_pool)[:, 1]

# Cầm predicted_scores này gắn ngược lại vào danh sách bài báo rồi dùng .sort() từ cao xuống thấp là ra bảng xếp hạng LTR chuẩn chỉ!

Bắt đầu thả xích cho Gradient Boosting học luật xếp hạng...
      Iter       Train Loss   Remaining Time 
         1           0.8175            3.44s
         2           0.8154            3.59s
         3           0.8137            3.72s
         4           0.8120            3.58s
         5           0.8106            3.62s
         6           0.8093            3.45s
         7           0.8080            3.28s
         8           0.8071            3.21s
         9           0.8060            3.12s
        10           0.8052            3.05s
        20           0.7981            3.28s
        30           0.7921            3.10s
        40           0.7865            3.05s
        50           0.7804            2.84s
        60           0.7754            2.79s
        70           0.7703            2.72s
        80           0.7659            2.82s
        90           0.7613            2.74s
       100           0.7566            2.65s
       200           0.7188            

In [21]:
ltr_model.predict_proba(data_tfidf)[:,1]

array([0.43465006, 0.06347037, 0.08104296, ..., 0.12884267, 0.13413599,
       0.11037886], shape=(7000,))

In [22]:
import time

def evaluate(embedding_matrix, top_k=10):
    users_id, target_dict, _, users_vector, history = load_user()
    num_val_users = len(users_id)
    num_papers_total = embedding_matrix.shape[0]
    
    mrr_list = []          # Lưu MRR toàn cục
    mrr_at_k_list = []     # Lưu MRR@10
    
    print(f"🚀 Bắt đầu đo lường ONLY MRR@{top_k}...")
    start_time = time.time()
    
    # Ép mảng category tĩnh để vector hóa
    all_categories_array = df['primary_category'].values
    
    for i, user in enumerate(users_id):
        user_his_cat = history_set(user, history)
        user_vec = users_vector[i].reshape(1, -1)
        
        # 1. TÍNH ĐIỂM VÀ ĐẶC TRƯNG
        cos_scores = cossin(user_vec, embedding_matrix).flatten()
        rec_scores = reccency_scores 
        cat_match_scores = np.isin(all_categories_array, list(user_his_cat)).astype(np.int16)
        
        user_features = np.column_stack((cos_scores, rec_scores, cat_match_scores))
        
        # 2. DỰ ĐOÁN XÁC SUẤT 
        user_preds = ltr_model.predict_proba(user_features)[:, 1]
        
        # 3. TÌM VỊ TRÍ BÀI TARGET
        target_idx = -1
        real_target_paper = target_dict[user]
        if real_target_paper in reverse_mapping:
            target_idx = reverse_mapping[real_target_paper]
            
        # 4. TÍNH MRR
        if target_idx != -1:
            # Sắp xếp index các bài báo theo xác suất từ cao xuống thấp
            ranked_indices = np.argsort(user_preds)[::-1]
            
            # Khảo sát xem bài Target đang nằm ở vị trí thứ mấy (Cộng 1 vì index bắt đầu từ 0)
            rank = np.where(ranked_indices == target_idx)[0][0] + 1
            
            # Lưu MRR Toàn cục
            mrr_list.append(1.0 / rank)
            
            # Lưu MRR@10 (Ngoài Top 10 = 0 điểm)
            if rank <= top_k:
                mrr_at_k_list.append(1.0 / rank)
            else:
                mrr_at_k_list.append(0.0)
                
        if (i + 1) % 100 == 0:
            print(f"⏳ Đã xử lý {i + 1}/{num_val_users} users... ({(time.time() - start_time):.2f}s)")

    # 5. IN KẾT QUẢ
    print(f"\n✅ KẾT QUẢ ĐÁNH GIÁ MRR TRÊN {len(mrr_list)} USERS:")
    print("-" * 40)
    print(f"🎯 Mean MRR (Toàn cục) : {np.mean(mrr_list):.4f}")
    print(f"🎯 Mean MRR@{top_k}      : {np.mean(mrr_at_k_list):.4f}")
    print("-" * 40)

# --- Kích hoạt chạy với SBERT ---
# evaluate_mrr_only(sbert_embeddings, top_k=10)

In [23]:
# evaluate(tfidf_matrix)
# #evaluate(sbert_embeddings)